In [0]:
%run ./_utils

In [ ]:
import json
from datetime import datetime
from pyspark.sql import functions as F

#### Create `openalex.common.openalex_concepts_snapshot` in same format as API

In [ ]:
df_transformed = (
    spark.read.table("openalex.common.concepts_api")
    # Convert BIGINT id to full URL
    .withColumn("id", F.concat(F.lit("https://openalex.org/C"), F.col("id")))
)

df_transformed.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("openalex.common.openalex_concepts_snapshot")

#### Export to JSONL + Parquet on S3
Writes both formats under `s3://openalex-snapshots/full/{date}/{format}/concepts/`.

In [0]:
date_str = get_snapshot_date()
df = spark.read.table("openalex.common.openalex_concepts_snapshot")
export_partitioned_all_formats(spark, dbutils, df, date_str, "concepts", salt=False)